# 06 -- Runaway Agent Protection Demo

Implements the step-count/timeout ceiling protection described in Chapter 7's "runaway-agent
protection" section and the routing-bug incident writeup in that same chapter: a mocked Supervisor
loop that can route back to the same agent indefinitely if its own completion-check logic has a bug,
demonstrated first **WITHOUT** the protection (capped at a low, safe iteration limit purely so this
notebook doesn't actually loop forever), and then **WITH** a step-count ceiling that cleanly terminates
the run and flags the case for human review instead of looping.

This is the same failure shape as the real incident Chapter 7 describes: a routing check that misread a
correctly-empty result as "incomplete" and kept re-invoking the same agent, burning a real LLM call
each time, with no forward progress. Entirely offline -- the "agent" below is a deterministic mock
function, and the routing bug is reproduced deliberately, not simulated by chance. No real API keys, no
network calls, no external LLM SDKs -- standard library plus `pandas` only.

In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
print("Environment ready. Offline, deterministic, no network calls.")

Environment ready. Offline, deterministic, no network calls.


## 1. The buggy agent and the buggy completion check

`compliance_agent_call` mimics the real incident from Chapter 7: for a borrower whose industry
genuinely has no extra covenant requirements, the Credit Policy Compliance Agent correctly returns an
**empty** `policy_citations` list -- that's a valid, complete result, not a failure.

`is_complete_BUGGY` is the routing bug: it treats an empty list as "not complete yet," so a Supervisor
using this check will call the agent again, get the same valid empty result again, and call it again --
forever, since the input never changes and the agent is deterministic.

In [2]:
def compliance_agent_call(application, call_log):
    """Deterministic mock: this borrower's industry genuinely has no extra covenants to cite, so an
    empty policy_citations list is a CORRECT, complete result -- not a sign anything went wrong."""
    call_log.append(application["application_id"])
    return {"agent": "credit_policy_compliance", "policy_citations": [], "status": "ok"}


def is_complete_BUGGY(agent_result):
    """THE BUG: treats an empty citations list as incomplete, indistinguishable from a malformed or
    missing response. This is exactly the routing bug described in Chapter 7 -- it can't tell
    "correctly found nothing to cite" apart from "didn't finish."""
    return len(agent_result["policy_citations"]) > 0


def is_complete_FIXED(agent_result):
    """THE FIX: check an explicit status field the agent sets itself, rather than inferring
    completeness from whether a list happened to be non-empty."""
    return agent_result.get("status") == "ok"


SYNTHETIC_APPLICATION = {"application_id": "LOAN-2026-0512", "industry": "no_extra_covenants_industry"}

print("Buggy and fixed completion checks defined.")

Buggy and fixed completion checks defined.


## 2. WITHOUT protection: the Supervisor loops

A minimal Supervisor loop that keeps re-invoking the Compliance Agent as long as `is_complete_BUGGY`
says the result isn't complete yet. With no step-count ceiling, this would loop **forever** on this
input, since the agent is deterministic and the bug never resolves on its own -- exactly the real
incident from Chapter 7, where the loop ran for a long, unbounded stretch of real time before it was
manually killed.

To demonstrate the problem safely in a notebook (without actually hanging), this cell caps the loop at
a deliberately low `UNSAFE_DEMO_CAP` purely as a notebook-safety measure -- **not** a step-count
ceiling the Supervisor itself is aware of or designed around. The point of this section is that the
loop hits that external safety cap still reporting `still_incomplete`, having made zero forward
progress across every iteration -- proof that nothing internal to this loop would ever have stopped it.

In [3]:
UNSAFE_DEMO_CAP = 25  # notebook-safety cap only -- NOT a step-count ceiling the loop itself knows about

call_log_unsafe = []
iterations = 0
result = None

while iterations < UNSAFE_DEMO_CAP:
    result = compliance_agent_call(SYNTHETIC_APPLICATION, call_log_unsafe)
    iterations += 1
    if is_complete_BUGGY(result):
        break
    # is_complete_BUGGY never returns True for this input -- the loop has no other exit condition.

print(f"Loop ran {iterations} iterations before hitting the notebook-safety cap "
      f"({UNSAFE_DEMO_CAP}), with zero awareness of a step limit.")
print(f"Real LLM calls that would have been burned: {len(call_log_unsafe)}")
print(f"Final is_complete_BUGGY(result): {is_complete_BUGGY(result)}")

assert iterations == UNSAFE_DEMO_CAP, (
    "Expected the unprotected loop to run every single iteration up to the safety cap -- "
    "it never found a way to terminate itself."
)
assert is_complete_BUGGY(result) is False, (
    "Expected the buggy completion check to still report incomplete even after many iterations -- "
    "no amount of retrying fixes a routing bug."
)
print("\nCONFIRMED: without a step-count ceiling, this loop never terminates on its own. "
      "Every iteration is a wasted, real LLM call with zero forward progress -- the exact shape of "
      "the incident Chapter 7 describes, where the fix that should have existed from day one was a "
      "hard ceiling the Supervisor enforces independently of its own routing logic.")

Loop ran 25 iterations before hitting the notebook-safety cap (25), with zero awareness of a step limit.
Real LLM calls that would have been burned: 25
Final is_complete_BUGGY(result): False

CONFIRMED: without a step-count ceiling, this loop never terminates on its own. Every iteration is a wasted, real LLM call with zero forward progress -- the exact shape of the incident Chapter 7 describes, where the fix that should have existed from day one was a hard ceiling the Supervisor enforces independently of its own routing logic.


## 3. WITH protection: a step-count ceiling that terminates cleanly

The fix from Chapter 7: a hard step-count limit (and, in production, a wall-clock timeout alongside it)
enforced by the Supervisor's own control logic, **independent of whatever the routing/completion check
decides**. When the limit is hit, the run terminates immediately with an explicit
`escalation_reason: step_limit_exceeded` flag -- distinct from a normal completion or a normal
escalation, so it's separately monitorable -- and the case is handed to a human for review instead of
being allowed to keep looping.

In [4]:
class StepLimitExceeded(Exception):
    """Raised when the orchestration graph hits its hard step-count ceiling."""


def run_supervisor_with_ceiling(application, is_complete_fn, agent_fn, max_steps):
    """Mirrors a real Supervisor's dispatch loop, but bounded by a hard step-count ceiling that the
    Supervisor enforces itself -- completely independent of the (possibly buggy) completion check."""
    call_log = []
    for step in range(1, max_steps + 1):
        result = agent_fn(application, call_log)
        if is_complete_fn(result):
            return {
                "status": "completed",
                "steps_used": step,
                "result": result,
            }
    # Step limit reached without the completion check ever being satisfied.
    return {
        "status": "escalated",
        "escalation_reason": "step_limit_exceeded",
        "steps_used": max_steps,
        "result": result,
    }


MAX_STEPS = 3  # a real deployment would tune this per agent; kept small here to show it firing quickly

# Run with the SAME buggy completion check as before -- the only thing that changed is the ceiling.
call_log_protected = []
outcome = run_supervisor_with_ceiling(
    SYNTHETIC_APPLICATION, is_complete_BUGGY, compliance_agent_call, MAX_STEPS
)

print("Outcome with the step-count ceiling in place, same buggy completion check:")
for k, v in outcome.items():
    print(f"  {k}: {v}")

assert outcome["status"] == "escalated"
assert outcome["escalation_reason"] == "step_limit_exceeded"
assert outcome["steps_used"] == MAX_STEPS
print(f"\nPASS: the loop terminated cleanly after exactly {MAX_STEPS} steps -- not indefinitely -- "
      "and was flagged for human review with a distinct, monitorable reason code, rather than being "
      "allowed to keep burning LLM calls forever.")

Outcome with the step-count ceiling in place, same buggy completion check:
  status: escalated
  escalation_reason: step_limit_exceeded
  steps_used: 3
  result: {'agent': 'credit_policy_compliance', 'policy_citations': [], 'status': 'ok'}

PASS: the loop terminated cleanly after exactly 3 steps -- not indefinitely -- and was flagged for human review with a distinct, monitorable reason code, rather than being allowed to keep burning LLM calls forever.


## 4. Confirm the ceiling doesn't get in the way of a genuinely valid case

The ceiling has to be a safety net, not a source of false escalations -- verify that once the routing
bug itself is fixed (using `is_complete_FIXED`, the explicit-status-field check from section 1), the
same application completes normally, well within the step budget, and is never escalated.

In [5]:
call_log_fixed = []
outcome_fixed = run_supervisor_with_ceiling(
    SYNTHETIC_APPLICATION, is_complete_FIXED, compliance_agent_call, MAX_STEPS
)

print("Outcome with the ROUTING BUG FIXED (explicit status field, not list-emptiness):")
for k, v in outcome_fixed.items():
    print(f"  {k}: {v}")

assert outcome_fixed["status"] == "completed"
assert outcome_fixed["steps_used"] == 1
print(f"\nPASS: with the actual routing bug fixed, the case completes correctly in a single step -- "
      "the step-count ceiling only fires when something is genuinely stuck, and doesn't interfere "
      "with a normal, valid completion.")

Outcome with the ROUTING BUG FIXED (explicit status field, not list-emptiness):
  status: completed
  steps_used: 1
  result: {'agent': 'credit_policy_compliance', 'policy_citations': [], 'status': 'ok'}

PASS: with the actual routing bug fixed, the case completes correctly in a single step -- the step-count ceiling only fires when something is genuinely stuck, and doesn't interfere with a normal, valid completion.


## Recap

This notebook reproduced the real incident shape from Chapter 7 end to end: a routing bug that
misreads a correctly-empty agent result as incomplete, causing an unbounded retry loop (section 2,
demonstrated safely with an external notebook cap standing in for what would otherwise be an
indefinite loop), and the fix -- a hard step-count ceiling enforced by the Supervisor independently of
its own routing logic, which turns an open-ended cost incident into a small, bounded, immediately
flagged one (section 3) -- without masking or interfering with genuinely valid cases once the
underlying bug is actually fixed (section 4). The lesson Chapter 7 draws from the real version of this
incident applies directly: the step-count ceiling is a basic safety property any supervisor-pattern
orchestration needs from day one, not a defense that's acceptable to add only after the first time it's
needed.